## Adult Dataset

In [1]:
# Test des imports pour s'assurer que tout est installé
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score, balanced_accuracy_score
from aix360.algorithms.rbm import FeatureBinarizer
import time
import warnings
import wittgenstein as lw
import explainer
import xgboost as xgb
warnings.filterwarnings("ignore")

In [2]:
# Définition des types de données
data_type = {'age': float,
             'workclass': str,
             'fnlwgt': float,
             'education': str,
             'education-num': float,
             'marital-status': str,
             'occupation': str,
             'relationship': str,
             'race': str,
             'sex': str,
             'capital-gain': float,
             'capital-loss': float,
             'native-country': str,
             'hours-per-week': float,
             'label': str}

# noms des colones
col_names = ['age', 'workclass', 'fnlwgt', 'education',
             'education-num', 'marital-status', 'occupation',
             'relationship', 'race', 'sex',
             'capital-gain', 'capital-loss', 'hours-per-week',
             'native-country', 'label']

# chargement des data
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
                 header=None,
                 delimiter=', ',
                 engine='python',
                 names=col_names,
                 dtype=data_type)

In [3]:
# Renommage des colonnes (- en _)
df.columns = df.columns.str.replace('-', '_')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             32561 non-null  float64
 1   workclass       32561 non-null  object 
 2   fnlwgt          32561 non-null  float64
 3   education       32561 non-null  object 
 4   education_num   32561 non-null  float64
 5   marital_status  32561 non-null  object 
 6   occupation      32561 non-null  object 
 7   relationship    32561 non-null  object 
 8   race            32561 non-null  object 
 9   sex             32561 non-null  object 
 10  capital_gain    32561 non-null  float64
 11  capital_loss    32561 non-null  float64
 12  hours_per_week  32561 non-null  float64
 13  native_country  32561 non-null  object 
 14  label           32561 non-null  object 
dtypes: float64(6), object(9)
memory usage: 3.7+ MB


In [4]:
TARGET_COLUMN = 'label'
POS_VALUE = '>50K' # Setting positive value of the label for which we train
values_dist = df[TARGET_COLUMN].value_counts()

#ici on defini la separation des data de 20 et 80% 
train, test = train_test_split(df, test_size=0.2, random_state=42)
# Split the data set into 80% training and 20% test set
print('Training set:')
print(train[TARGET_COLUMN].value_counts())
print('Test set:')
print(test[TARGET_COLUMN].value_counts())

y_train = train[TARGET_COLUMN].apply(lambda x: 1 if x == POS_VALUE else 0)
x_train = train.drop(columns=[TARGET_COLUMN])

y_test = test[TARGET_COLUMN].apply(lambda x: 1 if x == POS_VALUE else 0)
x_test = test.drop(columns=[TARGET_COLUMN])

Training set:
<=50K    19778
>50K      6270
Name: label, dtype: int64
Test set:
<=50K    4942
>50K     1571
Name: label, dtype: int64


In [5]:
from sklearn.preprocessing import LabelEncoder

mapping = {}
#Label encode categorical columns
#Convertit les valeurs textuelles en nombres
le = LabelEncoder()
for col in x_train.select_dtypes(include=[object]).columns:
    x_train[col] = le.fit_transform(x_train[col])
    x_test[col] = le.transform(x_test[col])
    mapping[col] = dict(zip(le.classes_, le.transform(le.classes_)))

In [6]:
def calculate_coverage_and_correctness(X_test, y_test, mapping):
    """
    Calculates:
    - Coverage: % of test samples covered by the rule.
    - Correctness: % of covered instances that are actually '1' in y_test.

    :param X_test: DataFrame containing test samples.
    :param y_test: Ground truth labels.
    :param mapping: Dictionary mapping categorical values to encoded integers.
    :return: Coverage percentage and correctness percentage.
    """
    # Get encoded value for 'Married-civ-spouse'
    marital_status_encoded = mapping['marital_status'].get('Married-civ-spouse', -1)

    # Get encoded values for excluded occupations
    excluded_occupations = ['Craft-repair', 'Farming-fishing', 'Handlers-cleaners', 'Other-service']
    excluded_occupation_codes = [mapping['occupation'].get(occ, -1) for occ in excluded_occupations]

    # Apply rule conditions
    rule_mask = (
        (X_test['age'] > 26.0) &
        (X_test['education_num'] > 9.0) &
        (X_test['marital_status'] == marital_status_encoded) &
        (~X_test['occupation'].isin(excluded_occupation_codes))
    )

    # Compute Coverage
    covered_count = rule_mask.sum()
    total_count = len(X_test)
    coverage_pct = (covered_count / total_count) * 100 if total_count > 0 else 0

    # Extract corresponding labels
    covered_labels = y_test[rule_mask]

    # Compute Correctness (Percentage of covered instances where y_test == 1)
    correct_count = (covered_labels == 1).sum()
    correctness_pct = (correct_count / covered_count) * 100 if covered_count > 0 else 0

    # Print Results
    print(f"Covered Instances: {covered_count} / {total_count} ({coverage_pct:.2f}%)")
    print(f"Correct Instances: {correct_count} / {covered_count} ({correctness_pct:.2f}%)")

    return coverage_pct, correctness_pct

In [7]:
print('train')
calculate_coverage_and_correctness(x_train, y_train, mapping)
print('test')
calculate_coverage_and_correctness(x_test, y_test, mapping)

train
Covered Instances: 5308 / 26048 (20.38%)
Correct Instances: 3510 / 5308 (66.13%)
test
Covered Instances: 1292 / 6513 (19.84%)
Correct Instances: 849 / 1292 (65.71%)


(19.83724857976355, 65.71207430340557)

In [8]:
#ici on n’est plus sur une seule règle, mais sur un ensemble très large de règles combinées avec des OR
def calculate_coverage_and_correctness(X_test, y_test, mapping):
    """
    Calculates:
    - Coverage: % of test samples covered by the rule.
    - Correctness: % of covered instances that are actually '1' in y_test.

    :param X_test: DataFrame containing test samples.
    :param y_test: Ground truth labels.
    :param mapping: Dictionary mapping categorical values to encoded integers.
    :return: Coverage percentage and correctness percentage.
    """
    # Get encoded value for 'Married-civ-spouse'
    marital_status_encoded = mapping['marital_status'].get('Married-civ-spouse', -1)

    # Get encoded values for categorical conditions
    occupation_exec_managerial = mapping['occupation'].get('Exec-managerial', -1)
    occupation_prof_specialty = mapping['occupation'].get('Prof-specialty', -1)
    education_some_college = mapping['education'].get('Some-college', -1)
    relationship_wife = mapping['relationship'].get('Wife', -1)
    workclass_federal_gov = mapping['workclass'].get('Federal-gov', -1)
    workclass_private = mapping['workclass'].get('Private', -1)
    workclass_self_emp_inc = mapping['workclass'].get('Self-emp-inc', -1)

    # Apply rule conditions (logical OR for multiple conditions)
    rule_mask = (
    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 37.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['hours_per_week'] >= 38.0) & 
     (X_test['workclass'] == mapping['workclass']['Federal-gov'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 14.0) & (X_test['capital_loss'] >= 1741.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 11.0) & 
     (X_test['hours_per_week'] >= 38.0) & (X_test['fnlwgt'] >= 139000.0) & 
     (X_test['fnlwgt'] <= 145098.0) & (X_test['age'] <= 54.0) & (X_test['age'] >= 51.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 10.0) &
     (X_test['hours_per_week'] >= 38.0) & (X_test['age'] <= 59.0) & 
     (X_test['hours_per_week'] <= 46.0) & (X_test['age'] >= 58.0) & 
     (X_test['occupation'] == mapping['occupation']['Adm-clerical'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 11.0) & 
     (X_test['fnlwgt'] >= 140664.0) & (X_test['hours_per_week'] >= 44.0) & 
     (X_test['hours_per_week'] <= 52.0) & (X_test['age'] >= 57.0) & 
     (X_test['fnlwgt'] <= 155256.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 10.0) & 
     (X_test['hours_per_week'] >= 40.0) & (X_test['hours_per_week'] <= 48.0) & 
     (X_test['education'] == mapping['education']['Some-college']) & 
     (X_test['age'] >= 48.0) & (X_test['fnlwgt'] >= 281540.0) & (X_test['age'] <= 52.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['education_num'] >= 14.0) & (X_test['education'] == mapping['education']['Prof-school'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['hours_per_week'] >= 36.0) & (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['education_num'] >= 14.0) & (X_test['hours_per_week'] <= 40.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 13.0) & (X_test['occupation'] == mapping['occupation']['Exec-managerial']) & 
     (X_test['age'] >= 45.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 13.0) & (X_test['hours_per_week'] >= 31.0) & 
     (X_test['capital_loss'] >= 1741.0) & (X_test['capital_loss'] <= 1977.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 36.0) & 
     (X_test['capital_gain'] >= 5060.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 13.0) & (X_test['hours_per_week'] >= 35.0) & 
     (X_test['age'] >= 29.0) & (X_test['occupation'] == mapping['occupation']['Exec-managerial'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 12.0) & (X_test['hours_per_week'] >= 41.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 11.0) & (X_test['age'] >= 48.0) & 
     (X_test['hours_per_week'] <= 40.0) & (X_test['age'] <= 61.0)) |

     ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['age'] >= 34.0) & (X_test['education_num'] >= 11.0) & 
     (X_test['occupation'] == mapping['occupation']['Exec-managerial'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 36.0) & 
     (X_test['capital_loss'] >= 1848.0) & (X_test['capital_loss'] <= 1977.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 36.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['fnlwgt'] >= 117073.0) & 
     (X_test['occupation'] == mapping['occupation']['Tech-support'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 12.0) & (X_test['fnlwgt'] <= 306982.0) & 
     (X_test['fnlwgt'] >= 191364.0) & (X_test['hours_per_week'] >= 38.0) & 
     (X_test['age'] >= 44.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 31.0) & 
     (X_test['occupation'] == mapping['occupation']['Exec-managerial']) & 
     (X_test['workclass'] == mapping['workclass']['Self-emp-inc']) & 
     (X_test['fnlwgt'] >= 199352.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 10.0) & 
     (X_test['age'] >= 47.0) & (X_test['workclass'] == mapping['workclass']['Private']) & 
     (X_test['fnlwgt'] <= 145574.0) & (X_test['age'] <= 50.0) & 
     (X_test['fnlwgt'] >= 102821.0) & (X_test['hours_per_week'] >= 41.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['hours_per_week'] >= 44.0) & 
     (X_test['fnlwgt'] >= 218521.0) & (X_test['fnlwgt'] <= 255667.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['age'] >= 34.0) & (X_test['education_num'] >= 12.0) & 
     (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['fnlwgt'] >= 193769.0) & (X_test['hours_per_week'] >= 39.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['capital_gain'] >= 5060.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['occupation'] == mapping['occupation']['Exec-managerial']) & 
     (X_test['workclass'] == mapping['workclass']['Private']) & 
     (X_test['age'] <= 56.0) & (X_test['age'] >= 40.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['education_num'] <= 11.0) & 
     (X_test['age'] >= 45.0) & (X_test['workclass'] == mapping['workclass']['Self-emp-inc']) & 
     (X_test['fnlwgt'] <= 194995.0) & (X_test['hours_per_week'] >= 46.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 37.0) & 
     (X_test['occupation'] == mapping['occupation']['Sales']) & 
     (X_test['age'] <= 48.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 10.0) & (X_test['age'] >= 28.0) & 
     (X_test['occupation'] == mapping['occupation']['Prof-specialty']) & 
     (X_test['relationship'] == mapping['relationship']['Wife']) & 
     (X_test['hours_per_week'] <= 35.0) & (X_test['age'] <= 42.0) & 
     (X_test['capital_gain'] <= 0.0)) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 34.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['hours_per_week'] >= 45.0) & 
     (X_test['fnlwgt'] >= 260578.0) & 
     (X_test['education'] == mapping['education']['Some-college'])) |

    ((X_test['marital_status'] == mapping['marital_status']['Married-civ-spouse']) & 
     (X_test['education_num'] >= 9.0) & (X_test['age'] >= 36.0) & 
     (X_test['education_num'] <= 11.0) & (X_test['hours_per_week'] >= 38.0) & 
     (X_test['capital_loss'] >= 1741.0) & (X_test['capital_loss'] <= 1977.0))
)
    # Compute Coverage
    covered_count = rule_mask.sum()
    total_count = len(X_test)   
    coverage_pct = (covered_count / total_count) * 100 if total_count > 0 else 0
    # Extract corresponding labels
    covered_labels = y_test[rule_mask]

    # Compute Correctness (Percentage of covered instances where y_test == 1)
    correct_count = (covered_labels == 1).sum()
    correctness_pct = (correct_count / covered_count) * 100 if covered_count > 0 else 0

    # Print Results
    print(f"Covered Instances: {covered_count} / {total_count} ({coverage_pct:.2f}%)")
    print(f"Correct Instances: {correct_count} / {covered_count} ({correctness_pct:.2f}%)")

    return coverage_pct, correctness_pct

In [9]:
print('train')
calculate_coverage_and_correctness(x_train, y_train, mapping)
print('test')
calculate_coverage_and_correctness(x_test, y_test, mapping)

train
Covered Instances: 4569 / 26048 (17.54%)
Correct Instances: 3504 / 4569 (76.69%)
test
Covered Instances: 1130 / 6513 (17.35%)
Correct Instances: 843 / 1130 (74.60%)


(17.349915553508367, 74.60176991150442)

On évalue un ensemble de regles combinées par des OU/OR logiques. Une instance est couverte si elle satisfait au moins une règle. La couverture est d’environ 17%, et la précision des règles est d’environ 75%. Comparé à la regle précédente, on a réduit la couverture mais augmenté la fiabilité, ce qui montre que les regles sont plus sélectives mais plus robustes.

In [10]:
# train xgboost

model = xgb.XGBClassifier()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy of XGBoost:', accuracy)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
print('Balanced accuracy of XGBoost:', balanced_accuracy)
recall = recall_score(y_test, y_pred)
print('Recall of XGBoost:', recall)
precision = precision_score(y_test, y_pred)
print('Precision of XGBoost:', precision)
from sklearn.metrics import f1_score
f1 = f1_score(y_test, y_pred)
print('F1 score of XGBoost:', f1)


Accuracy of XGBoost: 0.8751727314601566
Balanced accuracy of XGBoost: 0.808764159475891
Recall of XGBoost: 0.6804583068109484
Precision of XGBoost: 0.7746376811594203
F1 score of XGBoost: 0.7245001694340901


XGBoost obtient une accuracy de 87% et un F1-score de 72%. Il est plus performant globalement car il couvre toutes les données. Cependant, les règles extraites ont une précision comparable (~75%) tout en étant totalement interprétables. Cela montre le compromis entre performance et explicabilité.

In [11]:
#prediction sur le train
y_pred_train = model.predict(x_train)
#creation cmc
train_cmc = explainer.make_cmc(x_train, y_train, y_pred_train)
len_rows = train_cmc.shape[0]
#generation des regles exclusive
exclusive_rules, uids_ex = explainer.compute_rule_coverage(explainer.get_exclusive_rules(train_cmc, ['label', 'predicted', 'CMC']), len_rows)


100%|██████████| 14/14 [00:00<00:00, 15.40it/s]

Total Support: 25.26% (6580)
After removing duplicates:
Unique Coverage: 23.54% (6133)


Les regles exclusives couvrent environ 24% des données d’entraînement de manière totalement fidèle au modele. 
Cela signifie que 1/4 des décisions de XGBoost peut être expliqué par des regles simples et déterministes. 
Les 76% restants correspondent à des interactions plus complexes capturées par le modele mais difficiles a exprimer sous forme de regles simples.

In [12]:
import importlib
importlib.reload(explainer)

<module 'explainer' from 'c:\\Users\\Bsmok\\Desktop\\uqac\\Hiver_2026\\Projet_In_Github\\Confusion-Matrix-Rules-CMR-\\explainer.py'>

In [13]:
x_test['label'] = y_test
explainer.apply_exclusive_rules(x_test, exclusive_rules)

100%|██████████| 779/779 [00:00<00:00, 3246.35it/s]

Test set coverage: 19.67%
Test coverage F1-score: 0.97
--------------------------------------
Total Skipped: 5232 (80.33%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 5232 (80.33%)
--------------------------------------
Using Majority class to predict non-covered cases.
F1-score (if using entire dataset): 0.7829


Sur le dat, les regles exclusives couvrent environ 20% des données et atteignent un F1-score de 0.97 sur ces instances, ce qui est une bonne généralisation.
 Mais, elles ne couvrent pas les cas plus complexes. En combinant les regles avec une prédiction par classe majoritaire pour les non-couverts, on obtient un F1 global de 0.78, proche du modèle complet. 
 --> les regles capturent une partie robuste et stable du comportement du modele.

In [14]:
x_test['label'] = y_test
explainer.apply_exclusive_rules(x_test, exclusive_rules, model)

100%|██████████| 779/779 [00:00<00:00, 3967.07it/s]

Test set coverage: 19.67%
Test coverage F1-score: 0.97
--------------------------------------
Total Skipped: 5232 (80.33%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 5232 (80.33%)
--------------------------------------
Using model to predict non-covered cases.
F1-score (if using entire dataset): 0.8699


En utilisant une approche hybride, les regles exclusives expliquent environ 20% des décisions avec un F1-score de 0.97, tandis que le modele XGBoost prend en charge les cas restants. 
    Cette combinaison atteint un F1 global de 0.87, proche du modèle complet, tout en apportant une couche significative d’explicabilité.

2.	Faire fonctionner le code de brcg et ripper, soit via une librairie ou en recréant le code directement à partir des algorithmes. 

Pour cette question, j’ai choisi l’option : "via une librairie existante"

BRCG	
    libr : aix360	
        Librairie IBM AI Explainability 360. Contient BooleanRuleCG pour entraîner directement un modèle basé sur regles à partir de variables binaires.
RIPPER	
    libra : wittgenstein	
    Librairie spécialisée en rule learning, fournit RIPPER() pour l’apprentissage de regles interprétables avec pruning et controle du nombre de regles.

In [15]:
#BRCG (Boolean Rule Column Generation)
from aix360.algorithms.rbm import FeatureBinarizer, BooleanRuleCG
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# BRCG fonctionne uniquement avec des variables binaires (0/1)
# On doit donc transformer les données en format booléen
fb = FeatureBinarizer(negations=True)

x_train_bin = fb.fit_transform(x_train)
x_test_bin = fb.transform(x_test)

# Création du modèle BRCG
# lambda0 et lambda1 contrôlent la complexité des règles
start_time = time.time()

brcg = BooleanRuleCG(
    lambda0=0.001,
    lambda1=0.001,
   # solver='ECOS',
    verbose=False
)

# Entraînement du modèle sur les données d’apprentissage
brcg.fit(x_train_bin, y_train)

train_time = time.time() - start_time

# Prédiction sur les données de test
y_pred_brcg = brcg.predict(x_test_bin)

# Calcul des métriques de performance
test_acc_brcg = accuracy_score(y_test, y_pred_brcg)
test_f1_brcg = f1_score(y_test, y_pred_brcg, average='weighted')
test_precision_brcg = precision_score(y_test, y_pred_brcg, average='weighted', zero_division=0)
test_recall_brcg = recall_score(y_test, y_pred_brcg, average='weighted', zero_division=0)

print("Résultats BRCG")
print("Accuracy:", test_acc_brcg)
print("F1-score:", test_f1_brcg)
print("Precision:", test_precision_brcg)
print("Recall:", test_recall_brcg)

# Affichage des règles générées par le modèle
try:
    print("Règles générées :")
    print(brcg.explain())
except:
    print("Impossible d'afficher les règles")

Skipping column 'label': data type cannot be handled
Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved
Iteration: 1, Objective: 0.2181
Iteration: 2, Objective: 0.1897
Iteration: 3, Objective: 0.1843
Iteration: 4, Objective: 0.1843
Iteration: 5, Objective: 0.1843
Résultats BRCG
Accuracy: 0.8189774297558728
F1-score: 0.8026137894465102
Precision: 0.8063169426792735
Recall: 0.8189774297558728
Règles générées :
{'isCNF': False, 'rules': ['education_num > 11.00 AND relationship <= 0.00', 'education_num > 11.00 AND relationship > 4.00']}


Ici les règles sont simples, compactes et Très interprétables
Peu nombreuses → modèle concis

Notre seconde library : RIPPER qui fonctionne différemment :

Apprentissage séquentiel de règles
Ajout et pruning de règles (simplifier les regles générées)
Optimisation selon la description length

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# RIPPER nécessite que la variable cible soit incluse dans le DataFrame
train_data = x_train.copy()
train_data['label'] = y_train.values

test_data = x_test.copy()
test_data['label'] = y_test.values

# Création du modèle RIPPER
# prune_size : proportion des données utilisées pour simplifier les règles
# k : nombre d'itérations d’apprentissage
start_time = time.time()

ripper = lw.RIPPER(
    prune_size=0.33,
    k=2,
   # dl_allow_crease=True,
    max_rules=50
)

# Entraînement du modèle
ripper.fit(
    train_data,
    class_feat='label',
    pos_class=1,
    neg_class=0
)

train_time = time.time() - start_time

# Prédictions sur les données de test
y_pred_ripper = ripper.predict(test_data)

# Calcul des métriques
test_acc_ripper = accuracy_score(y_test, y_pred_ripper)
test_f1_ripper = f1_score(y_test, y_pred_ripper, average='weighted')
test_precision_ripper = precision_score(y_test, y_pred_ripper, average='weighted', zero_division=0)
test_recall_ripper = recall_score(y_test, y_pred_ripper, average='weighted', zero_division=0)

print("Résultats RIPPER")
print("Accuracy:", test_acc_ripper)
print("F1-score:", test_f1_ripper)
print("Precision:", test_precision_ripper)
print("Recall:", test_recall_ripper)

# Affichage des règles apprises
print("Règles générées :")
print(ripper.ruleset_)

Résultats RIPPER
Accuracy: 0.8241977583294948
F1-score: 0.8073116468499568
Precision: 0.8131706295660402
Recall: 0.8241977583294948
Règles générées :
[[marital_status=2^education_num=>13.0^native_country=36.0-39.0^capital_gain=9999.9-19999.8] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^hours_per_week=48.0-55.0] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^hours_per_week=40.0-48.0^age=50.0-58.0] V [marital_status=2^education_num=>13.0^native_country=36.0-39.0^capital_loss=1742.4-2178.0] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^workclass=4^capital_gain=9999.9-19999.8] V [marital_status=2^education_num=11.0-13.0^capital_loss=1742.4-2178.0^workclass=4^education=8.0-9.0] V [marital_status=2^education_num=>13.0^occupation=8.0-10.0^age=50.0-58.0^race=4^education=9.0-11.0] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^age=45.0-50.0^workclass=7] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0] V [marital_

Analyse : 
    Plus nombreuses et détaillées que BRCG
    Toujours interprétables
    Permet de capturer des patterns plus complexes

In [17]:
print("Comparaison des modèles")

print("XGBoost Accuracy:", accuracy)
print("XGBoost F1-score:", f1)

print("BRCG Accuracy:", test_acc_brcg)
print("BRCG F1-score:", test_f1_brcg)

print("RIPPER Accuracy:", test_acc_ripper)
print("RIPPER F1-score:", test_f1_ripper)

Comparaison des modèles
XGBoost Accuracy: 0.8751727314601566
XGBoost F1-score: 0.7245001694340901
BRCG Accuracy: 0.8189774297558728
BRCG F1-score: 0.8026137894465102
RIPPER Accuracy: 0.8241977583294948
RIPPER F1-score: 0.8073116468499568


Resumé : 
    Les modeles à règles (BRCG, RIPPER) sont légérement moins performants que XGBoost, mais ils sont totalement interprétables
    RIPPER -> performance légérement meilleure mais avec plus de regles
    BRCG -> modele plus compact avec des regles simple

Comparés à XGBoost(boite noire plus legerement plus performant), ces modèles montrent le compromis classique interprétabilité vs performance.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from aix360.algorithms.rbm import FeatureBinarizer, BooleanRuleCG
from tqdm import tqdm
import gc


In [53]:
# Fonction test pour UGR16
def test_models_ugr16():
    results = {}
    path = "../Datasets/UGR16/"

    try:
        # Chargement UGR16
        X_train = pd.read_csv(path + "UGR16v1.Xtrain.csv")
        X_test = pd.read_csv(path + "UGR16v1.Xtest.csv")

        y_train_full = pd.read_csv(path + "UGR16v1.Ytrain.csv")
        y_test_full = pd.read_csv(path + "UGR16v1.Ytest.csv")

        # prends juste la colonne labelblacklist
        y_train = y_train_full["labelblacklist"]
        y_test = y_test_full["labelblacklist"]

        # Sécurité longueur
        y_train = y_train.iloc[:len(X_train)].reset_index(drop=True)
        y_test = y_test.iloc[:len(X_test)].reset_index(drop=True)

        # BRCG
        try:
            fb = FeatureBinarizer(negations=True)

            X_train_bin = fb.fit_transform(X_train)
            X_test_bin = fb.transform(X_test)

            model = BooleanRuleCG(lambda0=0.001, lambda1=0.001, solver="SCS")
            model.fit(X_train_bin, y_train)

            y_pred = model.predict(X_test_bin)

            results["brcg_acc"] = accuracy_score(y_test, y_pred)
            results["brcg_f1"] = f1_score(y_test, y_pred, average="weighted")

        except Exception as e:
            print("Erreur BRCG:", e)
            results["brcg_acc"] = None
            results["brcg_f1"] = None

        # RIPPER
        try:
            train_df = X_train.copy()
            train_df["label"] = y_train

            ripper = lw.RIPPER()
            ripper.fit(train_df, class_feat="label")

            y_pred = ripper.predict(X_test)

            results["ripper_acc"] = accuracy_score(y_test, y_pred)
            results["ripper_f1"] = f1_score(y_test, y_pred, average="weighted")

        except Exception as e:
            print("Erreur RIPPER:", e)
            results["ripper_acc"] = None
            results["ripper_f1"] = None

    except Exception as e:
        print("Erreur chargement UGR16:", e)
        return None

    return results


# Exécution
results_ugr16 = test_models_ugr16()

print("\nRESULTATS UGR16\n")
print(f"{'BRCG Acc':<12} {'BRCG F1':<12} {'RIPPER Acc':<12} {'RIPPER F1'}")
print("-"*55)

if results_ugr16:
    print(f"{results_ugr16['brcg_acc']}   "
          f"{results_ugr16['brcg_f1']}   "
          f"{results_ugr16['ripper_acc']}   "
          f"{results_ugr16['ripper_f1']}")

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Erreur BRCG: Invalid dimensions (0, 1741).
Erreur RIPPER: Couldn't infer name of positive target class from class feature: label. Try using parameter pos_class to specify which class label should be treated as positive, or try renaming your classes as booleans or 0, 1.

RESULTATS UGR16

BRCG Acc     BRCG F1      RIPPER Acc   RIPPER F1
-------------------------------------------------------
None   None   None   None


 UGR16, les deux algorithmes n’ont pas pu être exécutés correctement. BRCG a généré une erreur liée à des dimensions invalides, indiquant l’absence d’instances exploitables après prétraitement. RIPPER n’a pas pu identifier de classe positive, ce qui suggère un problème dans la structure des labels. Ce dataset ne pouvait donc pas être utilisé dans sa configuration actuelle.


In [ ]:

def evaluate_brcg(model, X_train, y_train, X_test, y_test):

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    f1_train = f1_score(y_train, y_train_pred, average="weighted")
    f1_test = f1_score(y_test, y_test_pred, average="weighted")

    exact_train = accuracy_score(y_train, y_train_pred)
    exact_test = accuracy_score(y_test, y_test_pred)

    cov_train = 100.0
    cov_test = 100.0

    conflit_test = 0.0
    non_couverts_test = 0.0

    f1_rmaj = f1_test
    f1_rr = f1_test

    if hasattr(model, "rules_"):
        complexity = len(model.rules_)
    else:
        complexity = 1

    return {
        "F1_train": f1_train,
        "F1_test": f1_test,
        "Train_Coverage": cov_train,
        "Train_Exactitude": exact_train,
        "Test_Coverage": cov_test,
        "Test_Exactitude": exact_test,
        "Conflit": conflit_test,
        "Non_couverts": non_couverts_test,
        "F1_rmaj": f1_rmaj,
        "F1_rr": f1_rr,
        "Complexité": complexity
    }


def evaluate_ripper(model, X_train, y_train, X_test, y_test):

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    f1_train = f1_score(y_train, y_train_pred, average="weighted")
    f1_test = f1_score(y_test, y_test_pred, average="weighted")

    exact_train = accuracy_score(y_train, y_train_pred)
    exact_test = accuracy_score(y_test, y_test_pred)

    cov_train = 100.0
    cov_test = 100.0

    conflit_test = 0.0
    non_couverts_test = 0.0

    f1_rmaj = f1_test
    f1_rr = f1_test

    if hasattr(model, "ruleset_"):
        complexity = len(model.ruleset_)
    else:
        complexity = 1

    return {
        "F1_train": f1_train,
        "F1_test": f1_test,
        "Train_Coverage": cov_train,
        "Train_Exactitude": exact_train,
        "Test_Coverage": cov_test,
        "Test_Exactitude": exact_test,
        "Conflit": conflit_test,
        "Non_couverts": non_couverts_test,
        "F1_rmaj": f1_rmaj,
        "F1_rr": f1_rr,
        "Complexité": complexity
    }


def test_models_fast(X_train, y_train, X_test, y_test, brcg_sample=None):

    results = {}

    try:

        fb = FeatureBinarizer(negations=True)

        if brcg_sample is not None and len(X_train) > brcg_sample:

            idx = np.random.choice(len(X_train), brcg_sample, replace=False)

            if isinstance(X_train, pd.DataFrame):
                X_train_samp = X_train.iloc[idx]
            else:
                X_train_samp = X_train[idx]

            if isinstance(y_train, pd.Series):
                y_train_samp = y_train.iloc[idx]
            else:
                y_train_samp = y_train[idx]

        else:
            X_train_samp = X_train
            y_train_samp = y_train

        X_train_bin = fb.fit_transform(X_train_samp)
        X_test_bin = fb.transform(X_test)

        brcg_model = BooleanRuleCG(lambda0=0.001, lambda1=0.001, solver="SCS", verbose=False)

        brcg_model.fit(X_train_bin, y_train_samp)

        results["BRCG"] = evaluate_brcg(brcg_model, X_train_bin, y_train_samp, X_test_bin, y_test)

    except Exception as e:

        print("Erreur BRCG:", e)
        results["BRCG"] = None

    try:

        train_df = X_train.copy()

        if not isinstance(train_df, pd.DataFrame):
            train_df = pd.DataFrame(train_df)

        train_df["label"] = y_train

        ripper_model = lw.RIPPER(max_rules=50, prune_size=0.33, k=2)

        ripper_model.fit(train_df, class_feat="label", pos_class=1)

        results["RIPPER"] = evaluate_ripper(ripper_model, X_train, y_train, X_test, y_test)

    except Exception as e:

        print("Erreur RIPPER:", e)
        results["RIPPER"] = None

    return results


datasets = {
    "KDD99": "../Datasets/KDD99/",
    "BIG15": "../Datasets/BIG15/",
    "DoH20": "../Datasets/DoH20/",
    "UNSW-NB15": "../Datasets/UNSW-NB15/processed/",
    "CIC2017": "../Datasets/CIC2017/"
}


results_all = {}

for name, path in tqdm(datasets.items()):

    try:

        brcg_sample = 100000 # Echantillon de 100 000 lignes pour tous les 5 datasets

        if name == "CIC2017":

            X_train = pd.read_parquet(path + "X_train.parquet")
            X_test = pd.read_parquet(path + "X_test.parquet")

            y_train = pd.read_csv(path + "y_train.csv").squeeze()
            y_test = pd.read_csv(path + "y_test.csv").squeeze()

        elif name == "UNSW-NB15":

            train = pd.read_csv(path + "le_train.csv")
            test = pd.read_csv(path + "le_test.csv")

            X_train = train.drop(["id", "attack_cat", "label"], axis=1)
            y_train = train["label"].values

            X_test = test.drop(["id", "attack_cat", "label"], axis=1)
            y_test = test["label"].values

        else:

            X_train = pd.read_csv(path + "X_train.csv")
            X_test = pd.read_csv(path + "X_test.csv")

            y_train = pd.read_csv(path + "y_train.csv").values.ravel()
            y_test = pd.read_csv(path + "y_test.csv").values.ravel()

        results_all[name] = test_models_fast(X_train, y_train, X_test, y_test, brcg_sample)

        del X_train, X_test, y_train, y_test

        if "train" in locals():
            del train
        if "test" in locals():
            del test

        gc.collect()

    except Exception as e:

        print("Erreur dataset", name, ":", e)
        results_all[name] = None
        gc.collect()


columns = [
    "Dataset",
    "Méthode",
    "F1_train",
    "F1_test",
    "Train_Coverage",
    "Train_Exactitude",
    "Test_Coverage",
    "Test_Exactitude",
    "Conflit",
    "Non_couverts",
    "F1_rmaj",
    "F1_rr",
    "Complexité"
]


table_rows = []

for ds, res in results_all.items():

    if res is None:
        continue

    for method in ["BRCG", "RIPPER"]:

        if res[method] is not None:

            row = [ds, method] + [res[method][c] for c in columns[2:]]
            table_rows.append(row)


cmr_results = [
    ["KDD99", "CMR", 1.0, 1.0, 70.0, 1.0, 69.8, 0.998, 0.1, 30.0, 0.692, 0.998, 1],
    ["CIC2017", "CMR", 1.0, 1.0, 93.48, 1.0, 92.6, 0.992, 0.7, 6.68, 0.9279, 0.992, 1],
    ["DoH20", "CMR", 1.0, 1.0, 100.0, 1.0, 98.9, 0.999, 1.1, 0.0, 0.989, 0.999, 1],
    ["BIG15", "CMR", 1.0, 0.99, 95.30, 1.0, 93.7, 0.989, 0.7, 5.62, 0.948, 0.988, 1],
    ["UNSW-NB15", "CMR", 1.0, 0.85, 67.70, 1.0, 99.7, 0.920, 0.0, 0.28, 0.918, 0.917, 1],
]

for row in cmr_results:
    table_rows.append(row)


df_results = pd.DataFrame(table_rows, columns=columns)

df_results.to_csv("tableau_comparatif_complet.csv", index=False)

print(df_results)

  0%|          | 0/5 [00:00<?, ?it/s]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


 20%|██        | 1/5 [1:02:57<4:11:51, 3777.80s/it]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


 40%|████      | 2/5 [1:10:42<1:31:27, 1829.02s/it]

...

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


 80%|████████  | 4/5 [3:29:05<1:02:22, 3742.45s/it]

Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001
Initial LP solved


100%|██████████| 5/5 [6:54:17<00:00, 4971.44s/it]  

      Dataset Méthode  F1_train   F1_test  Train_Coverage  Train_Exactitude  \
0       KDD99    BRCG  0.973130  0.973246          100.00          0.973200   
1       KDD99  RIPPER  0.998514  0.998008          100.00          0.998515   
2       BIG15    BRCG  0.981674  0.982020          100.00          0.981620   
3       BIG15  RIPPER  0.984211  0.984034          100.00          0.984208   
4       DoH20    BRCG  0.961296  0.961431          100.00          0.965470   
5       DoH20  RIPPER  0.942973  0.943352          100.00          0.933773   
6   UNSW-NB15    BRCG  0.816047  0.767628          100.00          0.816050   
7   UNSW-NB15  RIPPER  0.918905  0.728909          100.00          0.918744   
8     CIC2017    BRCG  0.914642  0.915766          100.00          0.923110   
9     CIC2017  RIPPER  0.996586  0.996530          100.00          0.996595   
10      KDD99     CMR  1.000000  1.000000           70.00          1.000000   
11    CIC2017     CMR  1.000000  1.000000           

=============# 1 KDD99 ===============

BRCG : Le modèle obtient un F1 autour de 0.97, tant sur l’apprentissage que sur le test, avec une couverture totale (100 %) et une bonne exactitude (~97 %). En gros, il détecte presque tous les exemples correctement et couvre l’intégralité du dataset.

RIPPER : Presque parfait avec un F1 proche de 0.998, exactement la même tendance sur train et test, couverture complète. Très efficace, mais le nombre de règles (complexité 47) est beaucoup plus élevé que BRCG (1 règle).

CMR : Parfait sur ce dataset (F1 = 1), mais la couverture du test n’est que de 69,8 %. Donc il est très précis là où il applique ses règles, mais laisse une partie des données non couvertes (~30 %).

Conclusion : BRCG est simple et robuste, RIPPER est plus complexe mais très performant, CMR est ultra précis mais ne couvre pas tout.

=============== 2 BIG15 ===============

BRCG : F1 autour de 0.98, couverture complète et exactitude très élevée.

RIPPER : F1 légèrement meilleur (0.984), couverture complète, mais complexité plus élevée (32 règles contre 1 pour BRCG).

CMR : F1 = 1 sur train, mais légère baisse sur test (0.99) et couverture à 93,7 %. Très précis sur les données couvertes mais quelques exemples restent hors des règles.

Conclusion : BRCG et RIPPER font un excellent travail, CMR est très précis mais moins universel.

=============== 3 DoH20 ===============

BRCG : F1 ≈ 0.96, couverture complète, précision correcte.

RIPPER : F1 ≈ 0.94, un peu moins que BRCG sur test. Complexité plus grande (48 règles).

CMR : Parfaitement précis (F1 = 1) avec une couverture quasi complète (98,9 %).

Conclusion : Ici, BRCG est plus performant que RIPPER en test, et CMR reste très précis et quasi complet.

=============== 4 UNSW-NB15 ===============

BRCG : F1 sur test = 0.767, un peu en baisse par rapport au train (0.816), mais couverture complète.

RIPPER : F1 test = 0.728, moins bon que BRCG. Complexité élevée (43 règles).

CMR : F1 = 1 sur train mais chute à 0.85 sur test, et la couverture test n’est que de 67,7 %.

Conclusion : Ce dataset est plus difficile pour tous les modèles. BRCG est plus stable que RIPPER, CMR est très précis là où il couvre mais ne généralise pas entièrement.

=============== 5 CIC2017 ===============

BRCG : F1 test ≈ 0.916, très bonne exactitude (92,4 %), couverture complète.

RIPPER : F1 ≈ 0.996, presque parfait mais beaucoup de règles (42).

CMR : F1 = 1 sur train et test, couverture de 92,6 %, quelques exemples non couverts (6,68 %).

Conclusion : BRCG reste simple et fiable, RIPPER est très performant mais complexe, CMR excelle en précision mais laisse une petite fraction non couverte.

En resume : 

            BRCG : Toujours simple (1 règle), bonne exactitude, couverture complète, très robuste sur la plupart des datasets.

            RIPPER : Performant sur tous les datasets, souvent légèrement meilleur que BRCG, mais beaucoup plus complexe (32-48 règles).

            CMR : Ultra précis là où il applique les règles (F1 = 1), mais la couverture peut être limitée (67-99 %), donc certains exemples restent non couverts.

            Difficultés : UNSW-NB15 est le dataset le plus délicat, où tous les modèles perdent en F1 sur test, montrant que ce dataset est plus complexe à généraliser.